## Handling missing values

In [1]:
import pandas as pd
import numpy as np

data = {
    'Name':   ['Sumed', 'Priya', 'Rahul', None,    'Arjun'],
    'Dept':   ['Data',  'HR',   None,    'Finance','Data' ],
    'Salary': [60000,  45000, 75000,  np.nan,  80000 ],
    'Exp':    [3,       np.nan, 5,       4,        None  ]
}
df = pd.DataFrame(data)
print(df)

    Name     Dept   Salary  Exp
0  Sumed     Data  60000.0  3.0
1  Priya       HR  45000.0  NaN
2  Rahul     None  75000.0  5.0
3   None  Finance      NaN  4.0
4  Arjun     Data  80000.0  NaN


 *Notice Exp column shows 3.0 instead of 3 — because NaN is a float, so Pandas converts the whole column to float to accommodate it. This is called type promotion.*

## Detecting missing values

In [3]:
# isnull() → returns True where value is NaN, False elsewhere
# isna()   → exact same thing, just an alias. Use either.
df.isnull()



,Name,Dept,Salary,Exp
0,False,False,False,False
1,False,False,False,True
2,False,True,False,False
3,True,False,True,False
4,False,False,False,True


In [4]:
# Count missing values per column
# isnull() gives True/False → sum() treats True as 1, False as 0
df.isnull().sum()



Name      1
Dept      1
Salary    1
Exp       2
dtype: int64

In [5]:
# Missing percentage per column — more useful in real analysis
(df.isnull().sum() / len(df)) * 100



Name      20.0
Dept      20.0
Salary    20.0
Exp       40.0
dtype: float64

In [6]:
# Check if ANY value is missing in entire df
df.isnull().any().any()   # returns single True or False



np.True_

In [7]:
# notnull() → opposite of isnull()
df.notnull()               # True where value EXISTS

,Name,Dept,Salary,Exp
0,True,True,True,True
1,True,True,True,False
2,True,False,True,True
3,False,True,False,True
4,True,True,True,False


*In real projects, first thing you do is **df.isnull().sum()** to get a snapshot of data quality. If a column has more than 50-60% missing → consider dropping it entirely.*

## DROPNA

In [9]:
# dropna() removes rows (or columns) that contain NaN

# Default: drop ANY row that has at least one NaN
df.dropna()



,Name,Dept,Salary,Exp
0,Sumed,Data,60000.0,3.0


In [10]:
# how='all' → drop row only if ALL values are NaN
df.dropna(how='all')



,Name,Dept,Salary,Exp
0,Sumed,Data,60000.0,3.0
1,Priya,HR,45000.0,NaN
2,Rahul,None,75000.0,5.0
3,None,Finance,NaN,4.0
4,Arjun,Data,80000.0,NaN


In [11]:
# subset → only check specific columns for NaN
# Drop rows where Salary OR Name is missing
df.dropna(subset=['Salary', 'Name'])



,Name,Dept,Salary,Exp
0,Sumed,Data,60000.0,3.0
1,Priya,HR,45000.0,NaN
2,Rahul,None,75000.0,5.0
4,Arjun,Data,80000.0,NaN


In [12]:
# thresh → keep rows that have AT LEAST N non-NaN values
df.dropna(thresh=3)   # keep row only if 3+ values exist



,Name,Dept,Salary,Exp
0,Sumed,Data,60000.0,3.0
1,Priya,HR,45000.0,NaN
2,Rahul,None,75000.0,5.0
4,Arjun,Data,80000.0,NaN


In [13]:
# Drop COLUMNS that have any NaN (axis=1)
df.dropna(axis=1)

""
0
1
2
3
4


*dropna() without subset can be too aggressive — it drops rows that are mostly fine but have one small gap. Always use subset to be precise about which columns matter.*

## FILLNA

In [14]:
# fillna() replaces NaN with a value you specify
# This is called IMPUTATION — a key data science concept

# Fill with a constant value
df['Dept'].fillna('Unknown')

# Fill numeric column with MEAN — most common strategy
mean_salary = df['Salary'].mean()
df['Salary'].fillna(mean_salary)

# Fill with MEDIAN — better when data has outliers
df['Salary'].fillna(df['Salary'].median())

# Fill with MODE — best for categorical columns
df['Dept'].fillna(df['Dept'].mode()[0])
# mode() returns a Series — [0] gets the first (most frequent) value

# Fill different columns with different values at once
df.fillna({
    'Salary': df['Salary'].mean(),
    'Dept':   'Unknown',
    'Exp':    df['Exp'].median()
})

,Name,Dept,Salary,Exp
0,Sumed,Data,60000.0,3.0
1,Priya,HR,45000.0,4.0
2,Rahul,Unknown,75000.0,5.0
3,None,Finance,65000.0,4.0
4,Arjun,Data,80000.0,4.0


## FFILL & BFILL (forward fill and backward fill)

In [15]:
# ffill → Forward Fill: copies value from PREVIOUS row
# bfill → Backward Fill: copies value from NEXT row
# WHEN to use: time series data, sensor readings, stock prices
# where missing value is best estimated by the nearest known value

temp_data = {
    'Hour': [1, 2, 3, 4, 5],
    'Temp': [36.5, np.nan, np.nan, 37.1, 36.8]
}
ts = pd.DataFrame(temp_data)

ts['Temp'].ffill()   # NaN at hour 2,3 → filled with 36.5 (prev value)
ts['Temp'].bfill()   # NaN at hour 2,3 → filled with 37.1 (next value)

0    36.5
1    37.1
2    37.1
3    37.1
4    36.8
Name: Temp, dtype: float64

In [16]:
ts

,Hour,Temp
0,1,36.5
1,2,NaN
2,3,NaN
3,4,37.1
4,5,36.8


*This is exactly how your PPG signal project would handle dropped sensor readings — if a pulse reading is missing at timestamp T, the most logical fill is the reading just before or after it.*

## Right strategy: when to drop vs fill

In [17]:
# This is a DECISION, not just code
# Interviewers test your thinking here, not just syntax

# RULE 1: If missing % is very high (>50%) → DROP the column
missing_pct = df.isnull().sum() / len(df)
cols_to_drop = missing_pct[missing_pct > 0.5].index
df = df.drop(columns=cols_to_drop)

# RULE 2: If column is critical (like ID, target variable) → DROP the ROW
df = df.dropna(subset=['Name'])

# RULE 3: Numeric column, no outliers → fill with MEAN
df['Salary'] = df['Salary'].fillna(df['Salary'].mean())

# RULE 4: Numeric column, HAS outliers → fill with MEDIAN
df['Exp'] = df['Exp'].fillna(df['Exp'].median())

# RULE 5: Categorical column → fill with MODE or 'Unknown'
df['Dept'] = df['Dept'].fillna(df['Dept'].mode()[0])

# RULE 6: Time series → ffill or bfill

*This decision-making process is what separates a junior who just calls dropna() blindly from a senior analyst who thinks about WHY data is missing and what the right fill strategy is for each column.*

## Checking after handling

In [18]:
# After handling missing data, always verify
# This is professional habit — never assume your fillna worked

df.isnull().sum()          # should show all zeros now
df.isnull().any().any()    # should return False
df.info()                   # Non-Null Count should match total rows

<class 'pandas.core.frame.DataFrame'>
Index: 4 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Name    4 non-null      object 
 1   Dept    4 non-null      object 
 2   Salary  4 non-null      float64
 3   Exp     4 non-null      float64
dtypes: float64(2), object(2)
memory usage: 160.0+ bytes


*Always end your missing data handling with a verification step. In a real project or interview case study, showing this habit signals professionalism.*

## Everything You Must Know Cold

**NaN vs 0 vs ""** — NaN means data doesn't exist. 0 and "" are real values. Never confuse them.

Type promotion — when a numeric column has NaN, Pandas converts the whole column to float because NaN is a float type. 

That's why 3 becomes 3.0.

**isnull() = isna()** — identical, just aliases. Use either. isnull().sum() is your first step in any real project.

**dropna(subset=[])** — always use subset to be surgical. Dropping by default wipes too many rows.

**fillna()** strategies — mean for numeric without outliers, median for numeric with outliers, mode for categorical, ffill/bfill for time series. This decision logic is what interviewers want to hear.

**mode()[0] — mode()** returns a Series, not a single value. You need [0] to extract the actual most frequent value.

Always verify after — df.isnull().sum() after handling. Professional habit.